# 26. Bringing circuits in from other frameworks

qb-compiler does not ask you to rewrite your circuits. It reads OpenQASM 3, Cirq and PennyLane as
well as Qiskit, converting into its own IR so the calibration-aware passes and the preflight checks
work the same whatever the circuit was written in.

Each reader is an optional extra, so you install only what you use:

```bash
pip install qb-compiler[qasm3]      # OpenQASM 3
pip install qb-compiler[cirq]       # Cirq
pip install qb-compiler[pennylane]  # PennyLane
```

This notebook reports which of those are present in the environment that produced the output
below, rather than assuming, so nothing here is a claim you cannot check.

In [1]:
import importlib.util

from qiskit import QuantumCircuit

from qb_compiler.ir.converters.qiskit_converter import from_qiskit

AVAILABLE = {
    name: importlib.util.find_spec(module) is not None
    for name, module in (
        ("qasm3", "qiskit_qasm3_import"),
        ("cirq", "cirq"),
        ("pennylane", "pennylane"),
    )
}
for name, present in AVAILABLE.items():
    print(f"  {name:<11} {'available' if present else 'not installed in this environment'}")

# One circuit, used as the source for every conversion below.
source = QuantumCircuit(3)
source.h(0)
source.cx(0, 1)
source.cx(1, 2)

ir = from_qiskit(source)
print(f"\nqb-compiler IR: {ir.n_qubits} qubits, {ir.gate_count} gates")

  qasm3       available
  cirq        available
  pennylane   not installed in this environment

qb-compiler IR: 3 qubits, 3 gates


## 1. OpenQASM 3

QASM is the interchange format most tooling can already emit, which makes it the widest door into
the compiler. The conversion goes both ways, so qb-compiler can also hand a circuit back out.

In [2]:
if AVAILABLE["qasm3"]:
    from qb_compiler.ir.converters.qasm3_converter import from_qasm3, to_qasm3

    qasm_text = to_qasm3(ir)
    print(qasm_text.strip())

    round_tripped = from_qasm3(qasm_text)
    print(f"\nread back: {round_tripped.n_qubits} qubits, {round_tripped.gate_count} gates")
    print(f"gate counts match: {round_tripped.gate_count == ir.gate_count}")
else:
    print("skipped: pip install qb-compiler[qasm3]")

OPENQASM 3.0;
include "stdgates.inc";
qubit[3] q;
h q[0];
cx q[0], q[1];
cx q[1], q[2];



read back: 3 qubits, 3 gates
gate counts match: True


## 2. Cirq

Cirq circuits use qubit objects rather than integer indices, so the reader maps them onto the
IR's register positions in a stable order.

In [3]:
if AVAILABLE["cirq"]:
    from qb_compiler.ir.converters.cirq_converter import from_cirq, to_cirq

    cirq_circuit = to_cirq(ir)
    print(cirq_circuit)

    round_tripped = from_cirq(cirq_circuit)
    print(f"\nread back: {round_tripped.n_qubits} qubits, {round_tripped.gate_count} gates")
    print(f"gate counts match: {round_tripped.gate_count == ir.gate_count}")
else:
    print("skipped: pip install qb-compiler[cirq]")

0: ───H───@───────
          │
1: ───────X───@───
              │
2: ───────────X───

read back: 3 qubits, 3 gates
gate counts match: True


## 3. PennyLane

PennyLane describes circuits as quantum functions rather than as objects, so the reader works
from a tape.

In [4]:
if AVAILABLE["pennylane"]:
    from qb_compiler.ir.converters.pennylane_converter import from_pennylane, to_pennylane

    tape = to_pennylane(ir)
    print(tape)
    round_tripped = from_pennylane(tape)
    print(f"\nread back: {round_tripped.n_qubits} qubits, {round_tripped.gate_count} gates")
else:
    print("skipped: pip install qb-compiler[pennylane]")
    print("The call is the same shape as the two above:")
    print("    from qb_compiler.ir.converters.pennylane_converter import from_pennylane, to_pennylane")

skipped: pip install qb-compiler[pennylane]
The call is the same shape as the two above:
    from qb_compiler.ir.converters.pennylane_converter import from_pennylane, to_pennylane


## 4. Moving between the two circuit types

qb-compiler has a public circuit type (`qb_compiler.QBCircuit`) that you build directly, and an
internal IR the passes operate on. Qiskit's `QuantumCircuit` is a third representation. Rather
than requiring you to track which function wants which, `any_to_qiskit` and
`any_to_compiler_circuit` accept all three and convert only when needed.

That is what lets `check_viability` and `QBCompiler.compile` take the same object: previously one
wanted a Qiskit circuit and the other a `QBCircuit`, so running both on one circuit raised.

In [5]:
from qb_compiler import QBCircuit, QBCompiler, any_to_compiler_circuit, any_to_qiskit, check_viability

qb_circ = QBCircuit(3)
qb_circ.h(0)
qb_circ.cx(0, 1)
qb_circ.cx(1, 2)
qb_circ.measure_all()

as_qiskit = any_to_qiskit(qb_circ)
as_qbcircuit = any_to_compiler_circuit(source)
print(f"QBCircuit    -> {type(as_qiskit).__name__}: {as_qiskit.num_qubits} qubits")
print(f"QuantumCircuit -> {type(as_qbcircuit).__name__}: {as_qbcircuit.n_qubits} qubits")
print()

# Both entry points, same object, either representation.
compiler = QBCompiler.from_backend("ibm_fez")
for label, circuit in (("QBCircuit", qb_circ), ("QuantumCircuit", any_to_qiskit(qb_circ))):
    viability = check_viability(circuit, backend="ibm_fez")
    compiled = compiler.compile(circuit)
    print(f"  {label:<15} viability={viability.status:<9} compiled depth={compiled.compiled_depth}")

QBCircuit    -> QuantumCircuit: 3 qubits
QuantumCircuit -> QBCircuit: 3 qubits



  QBCircuit       viability=VIABLE    compiled depth=11


  QuantumCircuit  viability=VIABLE    compiled depth=11


## 5. What the compiler can target

`qbc backends` prints the registry as a truth table. The `LIVE STATUS` column is deliberately
blunt: `live` means the path is exercised against the real device, `live-unvalidated` means the
code path exists but has not been proven against hardware. `DEPS` is whether the vendor SDK for
live calibration is installed here, and `FIXTURE` whether a bundled snapshot ships for offline use.

In [6]:
!qbc backends

BACKEND            PROVIDER    LIVE STATUS       DEPS   FIXTURE
--------------------------------------------------------------
ibm_fez            ibm         live              yes    yes
ibm_marrakesh      ibm         live              yes    no
ibm_torino         ibm         live              yes    yes
ionq_aria          ionq        live-unvalidated  yes    no
ionq_forte         ionq        live-unvalidated  yes    no
iqm_emerald        iqm         live-unvalidated  yes    no
iqm_garnet         iqm         live-unvalidated  yes    no
quantinuum_h2      quantinuum  live-unvalidated  yes    no
rigetti_ankaa      rigetti     live-unvalidated  yes    yes


## Summary

- OpenQASM 3, Cirq and PennyLane readers mean the compiler meets circuits where they are, each
  behind an optional extra so you install only what you use.
- `any_to_qiskit` and `any_to_compiler_circuit` remove the question of which circuit type a given
  entry point wants.
- `qbc backends` states what is genuinely validated against hardware and what is not.